In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install tensorflow==2.12.0 pillow scipy


INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.0/586.0 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.7/440.7 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 MB 8.8 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.17.2
    Uninsta

In [ ]:
import numpy as np
import os
from PIL import Image
from scipy import linalg
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

# ---------------------------------------------------------
# 1. FID Calculation Function (Stable Version)
# ---------------------------------------------------------
def calculate_fid(real_features, fake_features):
    # Calculate statistics
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)

    mu_fake = np.mean(fake_features, axis=0)
    sigma_fake = np.cov(fake_features, rowvar=False)

    # Compute squared difference
    ssdiff = np.sum((mu_real - mu_fake) ** 2)

    # Stabilized matrix sqrt calculation
    covmean = linalg.sqrtm(sigma_real @ sigma_fake + 1e-6 * np.eye(sigma_real.shape[0])).real

    # Final FID calculation
    return ssdiff + np.trace(sigma_real + sigma_fake - 2 * covmean)

# ---------------------------------------------------------
# 2. Image Processing Pipeline
# ---------------------------------------------------------
class FIDCalculator:
    def __init__(self):
        self.model = InceptionV3(include_top=False, pooling='avg', weights='imagenet')
        self.img_size = (299, 299)

    def load_image(self, path):
        """Load and preprocess a single image"""
        try:
            img = Image.open(path).convert('RGB')
            img = img.resize(self.img_size)
            return preprocess_input(np.array(img, dtype=np.float32))
        except Exception as e:
            print(f"Skipped {os.path.basename(path)}: {str(e)}")
            return None

    def get_features(self, img_array):
        """Get features for a single image"""
        return self.model.predict(np.expand_dims(img_array, axis=0), verbose=0)[0]

# ---------------------------------------------------------
# 3. Main Execution
# ---------------------------------------------------------
def main():
    # Initialize calculator
    fid_calc = FIDCalculator()

    # Path configuration
    base_dir = '/content/drive/MyDrive/eval'
    real_dir = os.path.join(base_dir, 'actual')
    fake_dir = os.path.join(base_dir, 'fake')

    # Process images in batches
    batch_size = 32
    real_features, fake_features = [], []

    for patient_id in range(400, 451):
        # Build paths
        real_path = os.path.join(real_dir, f'patient{patient_id:04d}_actual.png')
        fake_path = os.path.join(fake_dir, f'patient{patient_id:04d}_fake.png')

        # Process pair
        real_img = fid_calc.load_image(real_path)
        fake_img = fid_calc.load_image(fake_path)

        if real_img is not None and fake_img is not None:
            real_features.append(fid_calc.get_features(real_img))
            fake_features.append(fid_calc.get_features(fake_img))

    # Calculate FID
    if len(real_features) >= 2:
        fid_score = calculate_fid(np.array(real_features), np.array(fake_features))
        print(f'Final FID Score: {fid_score:.2f}')
    else:
        print('Error: Need at least 2 valid image pairs')

if __name__ == '__main__':
    # Verify execution mode
    print(f'Eager execution enabled: {tf.executing_eagerly()}')
    main()

Eager execution enabled: True
Final FID Score: 46.26


In [ ]:
import numpy as np
import os
from PIL import Image
from scipy import linalg
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

# ---------------------------------------------------------
# 1. FID Calculation Function (Stable Version)
# ---------------------------------------------------------
def calculate_fid(real_features, fake_features):
    # Calculate statistics
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)

    mu_fake = np.mean(fake_features, axis=0)
    sigma_fake = np.cov(fake_features, rowvar=False)

    # Compute squared difference
    ssdiff = np.sum((mu_real - mu_fake) ** 2)

    # Stabilized matrix sqrt calculation
    covmean = linalg.sqrtm(sigma_real @ sigma_fake + 1e-6 * np.eye(sigma_real.shape[0])).real

    # Final FID calculation
    return ssdiff + np.trace(sigma_real + sigma_fake - 2 * covmean)

# ---------------------------------------------------------
# 2. Image Processing Pipeline
# ---------------------------------------------------------
class FIDCalculator:
    def __init__(self):
        self.model = InceptionV3(include_top=False, pooling='avg', weights='imagenet')
        self.img_size = (299, 299)

    def load_image(self, path):
        """Load and preprocess a single image"""
        try:
            img = Image.open(path).convert('RGB')
            img = img.resize(self.img_size)
            return preprocess_input(np.array(img, dtype=np.float32))
        except Exception as e:
            print(f"Skipped {os.path.basename(path)}: {str(e)}")
            return None

    def get_features(self, img_array):
        """Get features for a single image"""
        return self.model.predict(np.expand_dims(img_array, axis=0), verbose=0)[0]

# ---------------------------------------------------------
# 3. Main Execution
# ---------------------------------------------------------
def main():
    # Initialize calculator
    fid_calc = FIDCalculator()

    # Path configuration
    base_dir = '/content/drive/MyDrive/eval'
    real_dir = os.path.join(base_dir, 'actual')
    fake_dir = os.path.join(base_dir, 'fake_dice')

    # Process images in batches
    batch_size = 32
    real_features, fake_features = [], []

    for patient_id in range(400, 451):
        # Build paths
        real_path = os.path.join(real_dir, f'patient{patient_id:04d}_actual.png')
        fake_path = os.path.join(fake_dir, f'patient{patient_id:04d}_fake.png')

        # Process pair
        real_img = fid_calc.load_image(real_path)
        fake_img = fid_calc.load_image(fake_path)

        if real_img is not None and fake_img is not None:
            real_features.append(fid_calc.get_features(real_img))
            fake_features.append(fid_calc.get_features(fake_img))

    # Calculate FID
    if len(real_features) >= 2:
        fid_score = calculate_fid(np.array(real_features), np.array(fake_features))
        print(f'Final FID Score: {fid_score:.2f}')
    else:
        print('Error: Need at least 2 valid image pairs')

if __name__ == '__main__':
    # Verify execution mode
    print(f'Eager execution enabled: {tf.executing_eagerly()}')
    main()

Eager execution enabled: True
87910968/87910968 [==============================] - 0s 0us/step
Final FID Score: 120.50
